In [ ]:
from foswec_flap_base import FOSWECFlapStudyBase

In [ ]:
study = FOSWECFlapStudyBase(wavefreq= 1/2,
                            amplitude=0.1,
)
filename = f"foswec_flap_WOT_f1_{study.f1:.3f}_nf_{study.nfreq}".replace(".", "p") + ".nc"

study.load_or_run_bem(filename)

In [ ]:
study.nfreq

In [ ]:
filename

In [ ]:
results = study.run_mass_sweep([0, 1/2, 1, 3/2], bem_cache_prefix="foswec_1_flap", use_bem_cache=False)

In [ ]:
table = study.results_table(results)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 1, figsize=(7, 4), sharex=True, constrained_layout=True)

for mass, data in results.items():
    wec_tdom = data["wec_tdom"]
    t = wec_tdom.time.values
    pos = wec_tdom["pos"].isel(realization=0, influenced_dof=0).values
    vel = wec_tdom["vel"].isel(realization=0, influenced_dof=0).values

    axs[0].plot(t, pos , label=f"mass={mass}kg")
    axs[1].plot(t, vel, label=f"mass={mass}")

axs[0].set_ylabel("pos [rad]")
axs[1].set_ylabel("vel [rad/s]")

[ax.grid(True, alpha=0.3) for ax in axs.flatten()]
axs[0].legend()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(7, 4), sharex=True, constrained_layout=True)

for mass, data in results.items():
    pto_tdom = data["pto_tdom"]
    t = pto_tdom.time.values
    pmech = pto_tdom["power"].sel(type="mech").isel(realization=0, dof=0).values
    pelec = pto_tdom["power"].sel(type="elec").isel(realization=0, dof=0).values

    axs[0].plot(t, pmech, label=f"mass={mass}")
    axs[1].plot(t, pelec, label=r"$P_{avg}=$" f"{(pelec.mean()):.2f}W")

axs[0].set_ylabel("mech power")
axs[0].grid(True, alpha=0.3)
axs[0].legend()

axs[1].set_xlabel("time [s]")
axs[1].set_ylabel("elec power")
axs[1].grid(True, alpha=0.3)
axs[1].legend()

In [ ]:

fig, axs = plt.subplots(5, 1, figsize=(7, 7), sharex=True, constrained_layout=True)

epower0 = -1*results[0]['summary']['opt_power']
for mass, data in results.items():
    smry = data['summary']
    axs[0].scatter(smry['mass_on_top'], -1*smry['opt_power'])
    axs[1].scatter(smry['mass_on_top'], (-1*smry['opt_power']/epower0 -1)*100 )
    axs[2].scatter(smry['mass_on_top'], smry['opt_stiffness'])
    axs[3].scatter(smry['mass_on_top'], data['constraint_metrics']['peak_torque_utilization']*100)
    axs[4].scatter(smry['mass_on_top'], data['constraint_metrics']['peak_rotation_utilization']*100)

[ax.grid() for ax in axs.flatten()]
axs[0].set_ylabel('Electrical \n Power [W]')
axs[1].set_ylabel('Power \n Improvement [%]')

axs[2].set_ylabel('Opt DT \n Stiffness [Nm/rad]')
axs[3].set_ylabel('Peak generator \n torque utilization [%]')
axs[4].set_ylabel('Peak rotation \n utilization [%]')

axs[-1].set_xlabel('Mass on top of flap [kg]')

axs[0].set_title(f'FOSWEC, reg wave f = {study.wavefreq:.2f} Hz, A = {study.amplitude:.2f}m ')

In [ ]:
data['constraint_metrics']

In [ ]:
#TODO, run for different wave frequencies..
# what is the optimal stiffness mass combo for each frequency? and compare to baseline, also very frequency

In [ ]:
from collections import OrderedDict
import pandas as pd

# User inputs
wavefreq_vec = [1/6, 1/5, 1/4, 1/3, 1/2, 0.75, 1.0]   # Hz
amplitude_vec = [0.025, 0.05, 0.075, 0.10, 0.15]      # m
mass_vec = [0, 1/2, 1, 3/2, 2, 2.5]

bem_cache_prefix = "foswec_1_flap"
use_bem_cache = False

# Main container:
# results_grid[(wavefreq, amplitude)] = {
#     "study": study,
#     "mass_results": ...,
#     "table": ...,
#     "summary_rows": ...
# }
results_grid = OrderedDict()

summary_rows = []

for wavefreq in wavefreq_vec:
    for amplitude in amplitude_vec:
        print(f"Running case: wavefreq={wavefreq:.3f} Hz, amplitude={amplitude:.3f} m")
        filename = (
            f"foswec_flap_WOT_f1_{study.f1:.3f}_nf_{study.nfreq}"
            .replace(".", "p") + ".nc"
        )
        study = FOSWECFlapStudyBase(
            wavefreq=wavefreq,
            amplitude=amplitude,
        )

        study_base = FOSWECFlapStudyBase(
            wavefreq=wavefreq,
            amplitude=amplitude,
            optimize_stiffness=False,
            fixed_stiffness=0.0,
        )
        study_base.load_or_run_bem(filename)
        baseline_results = study_base.run_mass_sweep([0])

        study.load_or_run_bem(filename)

        mass_results = study.run_mass_sweep(
            mass_vec,
            bem_cache_prefix=bem_cache_prefix,
            use_bem_cache=use_bem_cache,
        )

        table = study.results_table(mass_results)

        case_summary_rows = []
        for mass, data in mass_results.items():
            smry = data["summary"]
            cmet = data.get("constraint_metrics", {})

            row = {
                "wavefreq": wavefreq,
                "amplitude": amplitude,
                "mass_on_top": smry.get("mass_on_top", mass),
                "opt_power": smry.get("opt_power"),
                "opt_stiffness": smry.get("opt_stiffness"),
                "peak_torque_utilization": cmet.get("peak_torque_utilization"),
                "peak_rotation_utilization": cmet.get("peak_rotation_utilization"),
            }

            case_summary_rows.append(row)
            summary_rows.append(row)

        results_grid[(wavefreq, amplitude)] = {
            "study": study,
            "filename": filename,
            "mass_results": mass_results,
            "table": table,
            "summary_rows": case_summary_rows,
        }

summary_df = pd.DataFrame(summary_rows)

print("\nDone.")
print(f"Number of wave/amplitude cases: {len(results_grid)}")
print(f"Number of mass-sweep result rows: {len(summary_df)}")

summary_df

In [ ]:
def plot_df_res(plot_df, qnty):
    wavefreq_vals = sorted(plot_df["wavefreq"].unique())
    amplitude_vals = sorted(plot_df["amplitude"].unique())

    # -------------------------------------------------------
    # 1) Power vs mass, one subplot per wave frequency
    # -------------------------------------------------------
    fig, axs = plt.subplots(
        1, len(wavefreq_vals), 
        figsize=(2* len(wavefreq_vals),3 ),
        sharey=True,
        constrained_layout=True
    )

    if len(wavefreq_vals) == 1:
        axs = [axs]

    for ax, wf in zip(axs, wavefreq_vals):
        sub = plot_df[plot_df["wavefreq"] == wf]

        for amp in amplitude_vals:
            s = sub[sub["amplitude"] == amp].sort_values("mass_on_top")
            if len(s) == 0:
                continue
            ax.plot(
                s["mass_on_top"],
                s[qnty],
                marker="o",
                label=f"A={amp:.3f} m"
            )
        ax.set_title(f"wf = {wf:.3f} Hz")
        ax.grid(True, alpha=0.3)
        ax.set_xlabel("Mass on top[kg]")

    axs[0].set_ylabel(qnty)
    axs[-1].legend()

    axs[-1].set_xlabel("Mass on top of flap [kg]")
    plt.tight_layout()
    plt.show()


In [ ]:
improv_df = summary_df.copy()

# Convert to positive generated electrical power
improv_df["elec_power_W"] = -improv_df["opt_power"]
# Choose the baseline mass
baseline_mass = 0

# Baseline power for each (wavefreq, amplitude)
baseline_df = (
    improv_df[improv_df["mass_on_top"] == baseline_mass]
    [["wavefreq", "amplitude", "elec_power_W"]]
    .rename(columns={"elec_power_W": "baseline_power_W"})
)

# Merge baseline back onto all rows
improv_df = improv_df.merge(
    baseline_df,
    on=["wavefreq", "amplitude"],
    how="left"
)

# Percent improvement relative to baseline mass
improv_df["power_improvement_pct"] = (
    (improv_df["elec_power_W"] / improv_df["baseline_power_W"] - 1.0) * 100.0
)

improv_df

In [ ]:
# Copy and clean
plot_df = summary_df.copy()

# Convert to positive generated electrical power if opt_power is stored negative
plot_df["elec_power_W"] = -plot_df["opt_power"]

# Sort for cleaner plotting
plot_df = plot_df.sort_values(["wavefreq", "amplitude", "mass_on_top"]).reset_index(drop=True)

wavefreq_vals = sorted(plot_df["wavefreq"].unique())
amplitude_vals = sorted(plot_df["amplitude"].unique())


plot_df_res(plot_df, "elec_power_W")
plot_df_res(plot_df, "peak_rotation_utilization")
plot_df_res(improv_df, "power_improvement_pct")
plot_df_res(improv_df, "peak_rotation_utilization")


plot_df_res(plot_df, "opt_stiffness")

# -------------------------------------------------------
# 3) Best mass for each (wavefreq, amplitude)
# -------------------------------------------------------
idx = plot_df.groupby(["wavefreq", "amplitude"])["elec_power_W"].idxmax()
best_df = plot_df.loc[idx].sort_values(["wavefreq", "amplitude"]).reset_index(drop=True)

fig, axs = plt.subplots(2, 1, figsize=(7, 6), sharex=True, constrained_layout=True)

for amp in amplitude_vals:
    s = best_df[best_df["amplitude"] == amp].sort_values("wavefreq")
    if len(s) == 0:
        continue

    axs[0].plot(
        s["wavefreq"],
        s["mass_on_top"],
        marker="o",
        label=f"A={amp:.3f} m"
    )
    axs[1].plot(
        s["wavefreq"],
        s["elec_power_W"],
        marker="o",
        label=f"A={amp:.3f} m"
    )

axs[0].set_ylabel("Best Mass [kg]")
axs[0].grid(True, alpha=0.3)
axs[0].legend()

axs[1].set_xlabel("Wave Frequency [Hz]")
axs[1].set_ylabel("Best Electrical Power [W]")
axs[1].grid(True, alpha=0.3)
axs[1].legend()

plt.show()

In [ ]:
plot_df